In [7]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 

In [8]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [15]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.past_y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))
        self.current_y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))
        self.next_y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-65

            if ii+jj-1<0:
                self.past_y[ii] = \
                    ord('A')-65
            else:
                self.past_y[ii] = \
                    ord(data[ii+jj-1])-65

            self.current_y[ii] = \
                ord(data[ii+jj])-65       
            self.next_y[ii] = \
                ord(data[ii+jj+1])-65

        self.X = tnsr(self.X).long()
        self.past_y = tnsr(self.past_y).long()
        self.current_y = tnsr(self.current_y).long()
        self.next_y = tnsr(self.next_y).long()

    def __getitem__(self, index):
        return self.X[index], self.past_y[index], self.current_y[index], self.next_y[index]

    def __len__(self):
        return self.X.shape[0]

In [35]:
def path_finder_loss(past_hat, current_hat, next_hat, past_token, current_token, next_token):
    ce1 = F.cross_entropy(past_hat.view(-1, past_hat.size(-1)), past_token.view(-1))
    ce2 = F.cross_entropy(current_hat.view(-1, current_hat.size(-1)), current_token.view(-1))
    ce3 = F.cross_entropy(next_hat.view(-1, next_hat.size(-1)), next_token.view(-1))

    return (ce1+ce2+ce3)/3

In [ ]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded)
        out = self.linear(out[:,-1,:])

        return out, h  
    
class RNNDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded, h)
        return self.fc(out) 
    
class RNNAutoencoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.encoder = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        self.decoder = RNNDecoder(vocab_size, embedding_dim, hidden_size)
    
    def forward(self, x, decoder_input):
        next_token, h = self.encoder(x)
        decoder_output = self.decoder(decoder_input, h)
        return next_token, decoder_output

# class PathEncoder(nn.Module):
#     def __init__(self, vocab_size, embedding_dim, hidden_size):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embedding_dim)
#         self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
#         self.past_head = nn.Linear(hidden_size, vocab_size)
#         self.current_head = nn.Linear(hidden_size, vocab_size)
#         self.next_head = nn.Linear(hidden_size, vocab_size)

#     def forward(self, x, h=None, train=True):
#         embedded = self.embedding(x)
#         out, h = self.rnn(embedded, h)
#         next_token = self.next_head(out[:,-1,:])

#         if train:
#             past_token = self.past_head(out[:,-1,:])
#             current_token = self.current_head(out[:,-1,:])
#             return past_token, current_token, next_token, h
#         else:
#             return next_token, h


In [62]:
### initial training ###
total_samples = 30000
working_memory = 1
short_term_memory = 2
hidden_size = 30
vocab_size = 7
embedding_dim = 2
lr = 1e-3
test_acc = []

model = PathEncoder(vocab_size, embedding_dim, hidden_size)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95)
criterion = path_finder_loss

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
correct = np.zeros(1000,dtype=float)

for X, y1, y2, y3 in train_loader:
    optimizer.zero_grad()

    if total==0:
        y1_, y2_, y3_, h = model(X)
    else:
        # print('hi')
        y1_, y2_, y3_, h = model(X, h=hidden)

    loss = criterion(y1_, y2_, y3_, y1, y2, y3)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        hidden = h.clone()
        true_y = y3[0][0]
        estimated_y = y3_.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')
    

Iter : 1001, loss: 1.1052, accuracy: 0.2890
Iter : 2001, loss: 0.9876, accuracy: 0.3600
Iter : 3001, loss: 0.9006, accuracy: 0.5060
Iter : 4001, loss: 0.2684, accuracy: 0.5520
Iter : 5001, loss: 0.5480, accuracy: 0.6150
Iter : 6001, loss: 0.3358, accuracy: 0.6460
Iter : 7001, loss: 0.3437, accuracy: 0.6360
Iter : 8001, loss: 0.4845, accuracy: 0.6520
Iter : 9001, loss: 0.3461, accuracy: 0.6860
Iter : 10001, loss: 0.2282, accuracy: 0.6820
Iter : 11001, loss: 0.4146, accuracy: 0.6590
Iter : 12001, loss: 0.2847, accuracy: 0.6700
Iter : 13001, loss: 0.3968, accuracy: 0.6600
Iter : 14001, loss: 0.0926, accuracy: 0.6830
Iter : 15001, loss: 0.2645, accuracy: 0.6460
Iter : 16001, loss: 0.2271, accuracy: 0.6820
Iter : 17001, loss: 0.2321, accuracy: 0.6640
Iter : 18001, loss: 0.2144, accuracy: 0.6800
Iter : 19001, loss: 0.3152, accuracy: 0.6630
Iter : 20001, loss: 0.1970, accuracy: 0.6650
Iter : 21001, loss: 0.2134, accuracy: 0.6620
Iter : 22001, loss: 0.2141, accuracy: 0.6610
Iter : 23001, loss:

In [78]:
### initial training ###
total_samples = 30
data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
prev_h = torch.ones(1,1,hidden_size)
dis = []

for X, y1, y2, y3 in train_loader:

    with torch.no_grad():
        if total==0:
            y1_, y2_, y3_, h = model(X)
        else:
            y1_, y2_, y3_, h = model(X, h=h)
            # break

        # dis.append(cosine(prev_h[0][0], h[0][0]))
        print('train', cosine(prev_h[0][0], h[0][0]), tokens[y3[0][0]])
        total += 1
        prev_h = h.clone()


train 0.9900286253353704 A
train 0.7289490019731318 G
train 0.8740471381593287 F
train 0.6915929931008498 D
train 0.7578827799345 E
train 0.8606375210920834 G
train 1.1143755301493241 D
train 1.1599113313532592 E
train 1.2311114354698216 F
train 1.1891463553207007 G
train 0.8210971615373317 C
train 0.8450177730459166 B
train 0.832092003173858 A
train 0.7113105849707493 G
train 0.973914510376144 D
train 1.0944417264699933 E
train 1.2365966741834467 F
train 1.1935691099903627 G
train 0.8229219490105577 D
train 1.019418306201465 E
train 1.2241480952379145 F
train 1.1757035910668094 G
train 0.8147760838060235 D
train 1.0168006690757463 F
train 0.8306347781104311 E
train 0.6995306682991431 G
train 0.9744336467547522 F


In [76]:
h

tensor([[[-0.9574,  0.9996, -0.9973,  0.8452,  1.0000, -0.9777,  0.9735,
          -0.9780,  0.9925, -0.6578,  1.0000,  0.9635, -0.9999,  1.0000,
          -0.7636,  0.9418, -0.9988,  0.9620, -0.6112, -0.8491, -0.9950,
           0.9890, -0.7892, -0.9990, -0.7874, -0.9993,  0.9986, -0.9976,
          -0.9977, -0.8435]]])

In [77]:
prev_h

tensor([[[-0.9574,  0.9996, -0.9973,  0.8452,  1.0000, -0.9777,  0.9735,
          -0.9780,  0.9925, -0.6578,  1.0000,  0.9635, -0.9999,  1.0000,
          -0.7636,  0.9418, -0.9988,  0.9620, -0.6112, -0.8491, -0.9950,
           0.9890, -0.7892, -0.9990, -0.7874, -0.9993,  0.9986, -0.9976,
          -0.9977, -0.8435]]])

In [67]:
y1_.argmax()

tensor(2)

In [68]:
y2_.argmax()

tensor(1)

In [69]:
y3_.argmax()

tensor(0)